# 05 — Multiclass Inference Demo

This notebook is an independent, read-only consumer of the final Dry Bean artifacts. It validates lineage, bytes, runtime safety, fitted state, input semantics, class order, and probability alignment before demonstrating educational inference. It does not acquire data, train, refit, select models, access the held-out partition, recalculate evaluation evidence, or create a completion artifact.


## 1. Inference Context and Boundary

The final model is a seven-class educational snapshot. A successful demonstration proves independent consumption of the frozen model artifact; it does not establish production validity, operational feature availability, monitoring, deployment safety, or an API contract.


In [1]:
from __future__ import annotations

from copy import deepcopy
import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd


def discover_project_root() -> Path:
    configured = os.getenv("DATASET_STUDY_ROOT")
    candidates = ([Path(configured).expanduser()] if configured else []) + [
        Path.cwd(), *Path.cwd().parents
    ]
    for candidate in candidates:
        root = candidate.resolve()
        if (root / "pyproject.toml").is_file() and (
            root / "scripts" / "smoke_predict.py"
        ).is_file():
            return root
    raise RuntimeError("Dataset-study project root could not be discovered.")


PROJECT_ROOT = discover_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.finalize_model import (
    load_and_validate_final_model_handoff,
    load_and_validate_inference_bundle,
)
from scripts.smoke_predict import (
    InferenceInputError,
    current_runtime_versions,
    load_validated_inference_pipeline,
    multiclass_output_to_frame,
    normalize_inference_input,
    predict_multiclass,
    predict_multiclass_batch,
    validate_bundle_handoff_alignment,
    validate_inference_readiness,
    validate_model_artifact_before_load,
    validate_runtime_compatibility,
)

HANDOFF_PATH = Path("artifacts/models/dry-bean/final-model-handoff.json")
BUNDLE_PATH = Path("artifacts/models/dry-bean/inference-bundle.json")

print({"project_root": ".", "handoff": HANDOFF_PATH.as_posix(), "bundle": BUNDLE_PATH.as_posix()})


{'project_root': '.', 'handoff': 'artifacts/models/dry-bean/final-model-handoff.json', 'bundle': 'artifacts/models/dry-bean/inference-bundle.json'}


## 2. Independent Final-Artifact Loading

The schema-aware finalization loaders validate the v2 handoff, bundle, complete sibling set, declared byte hashes, and model artifact hash. No preparation or model-selection DataFrame is loaded.


In [2]:
handoff = load_and_validate_final_model_handoff(
    project_root=PROJECT_ROOT,
    handoff_path=HANDOFF_PATH,
)
bundle = load_and_validate_inference_bundle(
    project_root=PROJECT_ROOT,
    bundle_path=BUNDLE_PATH,
)
validate_inference_readiness(handoff, bundle)
validate_bundle_handoff_alignment(handoff, bundle)

print({
    "handoff_schema": handoff["schema_version"],
    "bundle_schema": bundle["schema_version"],
    "dataset_slug": bundle["dataset_slug"],
    "problem_type": bundle["problem_type"],
    "feature_count": len(bundle["feature_columns"]),
    "class_count": len(bundle["output_class_order"]),
})


{'handoff_schema': 'final-model-handoff.v2', 'bundle_schema': 'inference-bundle.v2', 'dataset_slug': 'dry-bean', 'problem_type': 'multiclass_classification', 'feature_count': 16, 'class_count': 7}


## 3. Artifact Trust and Runtime Gates

`trusted_source=True` below is an explicit decision about this lineage-validated artifact after exact SHA-256 verification. It is not permission to load an arbitrary joblib. Exact compatibility is reported for reproducibility; load-safe compatibility is the mandatory deserialization gate.


In [3]:
validated_model_path = validate_model_artifact_before_load(
    project_root=PROJECT_ROOT,
    bundle=bundle,
    handoff=handoff,
)
expected_runtime = bundle["runtime_version_requirements"]
observed_runtime = current_runtime_versions()
exact_runtime_report = validate_runtime_compatibility(
    expected_runtime,
    observed_versions=observed_runtime,
    mode="exact",
)
load_safe_runtime_report = validate_runtime_compatibility(
    expected_runtime,
    observed_versions=observed_runtime,
    mode="load_safe",
    raise_on_incompatible=False,
)

print({
    "model_artifact_path": validated_model_path.relative_to(PROJECT_ROOT).as_posix(),
    "model_sha256_validated": bundle["model_artifact_sha256"],
    "expected_runtime": expected_runtime,
    "observed_runtime": observed_runtime,
    "exact": exact_runtime_report.as_dict(),
    "load_safe": load_safe_runtime_report.as_dict(),
})


{'model_artifact_path': 'artifacts/models/dry-bean/final-pipeline.joblib', 'model_sha256_validated': 'b7d8aa9c5846237c58d4e9cd05ed451cad3d36931822d1b74da28c3d8db1e478', 'expected_runtime': {'joblib': '1.5.3', 'pandas': '3.0.5', 'python': '3.13.13', 'scikit_learn': '1.9.0'}, 'observed_runtime': {'python': '3.13.13', 'pandas': '3.0.5', 'scikit_learn': '1.9.0', 'joblib': '1.5.3'}, 'exact': {'mode': 'exact', 'compatible': True, 'components': [{'component': 'python', 'expected': '3.13.13', 'observed': '3.13.13', 'compatible': True, 'status': 'compatible', 'detail': 'exact match'}, {'component': 'pandas', 'expected': '3.0.5', 'observed': '3.0.5', 'compatible': True, 'status': 'compatible', 'detail': 'exact match'}, {'component': 'scikit_learn', 'expected': '1.9.0', 'observed': '1.9.0', 'compatible': True, 'status': 'compatible', 'detail': 'exact match'}, {'component': 'joblib', 'expected': '1.5.3', 'observed': '1.5.3', 'compatible': True, 'status': 'compatible', 'detail': 'exact match'}], 'w

In [4]:
pipeline, loaded_handoff, loaded_bundle, runtime_report = (
    load_validated_inference_pipeline(
        project_root=PROJECT_ROOT,
        handoff_path=HANDOFF_PATH,
        bundle_path=BUNDLE_PATH,
        trusted_source=True,
    )
)
assert runtime_report.compatible is True
assert loaded_bundle["model_state_fingerprint"] == loaded_handoff["model_state_fingerprint"]

print({
    "trusted_model_reload_completed": True,
    "runtime_load_safe": runtime_report.compatible,
    "model_state_fingerprint_validated": loaded_bundle["model_state_fingerprint"],
    "pipeline_steps": list(pipeline.named_steps),
})


{'trusted_model_reload_completed': True, 'runtime_load_safe': True, 'model_state_fingerprint_validated': '0f301110a38db2e60d3c41432c69cc7458e301fd8865858f3ac9b853b87909aa', 'pipeline_steps': ['preprocess', 'model']}


## 4. Multiclass Input Contract

All input requirements come from the bundle. The caller may supply a mapping, Series, or DataFrame in any column order; normalization defensively copies and reorders it. Missing, duplicate, prohibited, unexpected, nonnumeric, and non-finite values are rejected. The final pipeline declares no learned imputation and no categorical features.


In [5]:
input_contract = pd.DataFrame(loaded_bundle["expected_input_schema"])
display(input_contract)
print({
    "required_input_columns": loaded_bundle["required_input_columns"],
    "prohibited_input_columns": loaded_bundle["prohibited_input_columns"],
    "missing_value_policy": loaded_bundle["missing_value_policy"],
    "categorical_features": loaded_bundle["categorical_features"],
})


,expected_dtype,missing_value_behavior,name,required,role
0,numeric,reject,Area,True,numerical
1,numeric,reject,Perimeter,True,numerical
2,numeric,reject,MajorAxisLength,True,numerical
3,numeric,reject,MinorAxisLength,True,numerical
4,numeric,reject,AspectRatio,True,numerical
5,numeric,reject,Eccentricity,True,numerical
6,numeric,reject,ConvexArea,True,numerical
7,numeric,reject,EquivDiameter,True,numerical
8,numeric,reject,Extent,True,numerical
9,numeric,reject,Solidity,True,numerical


{'required_input_columns': ['Area', 'Perimeter', 'MajorAxisLength', 'MinorAxisLength', 'AspectRatio', 'Eccentricity', 'ConvexArea', 'EquivDiameter', 'Extent', 'Solidity', 'Roundness', 'Compactness', 'ShapeFactor1', 'ShapeFactor2', 'ShapeFactor3', 'ShapeFactor4'], 'prohibited_input_columns': ['Class'], 'missing_value_policy': {'invalid_conversion_counts': {}, 'learned_imputation_in_final_pipeline': False, 'prepared_training_missing_value_count': 0, 'strategy': 'reject_missing_required_values'}, 'categorical_features': []}


## 5. Class-Order and Output Contract

The estimator and public output orders are intentionally inspected separately. Probability columns are resolved from the fitted estimator labels and then remapped into the bundle's output order before the educational prediction is selected.


In [6]:
estimator_order = list(pipeline.named_steps["model"].classes_)
output_order = loaded_bundle["output_class_order"]
assert estimator_order == loaded_bundle["estimator_class_order"]
assert set(estimator_order) == set(output_order)

display(pd.DataFrame({
    "estimator_class_order": pd.Series(estimator_order),
    "output_class_order": pd.Series(output_order),
}))
print(loaded_bundle["inference_output_contract"])


,estimator_class_order,output_class_order
0,BARBUNYA,SEKER
1,BOMBAY,BARBUNYA
2,CALI,BOMBAY
3,DERMASON,CALI
4,HOROZ,DERMASON
5,SEKER,HOROZ
6,SIRA,SIRA


{'binary_threshold': 'not_applicable', 'class_order': ['SEKER', 'BARBUNYA', 'BOMBAY', 'CALI', 'DERMASON', 'HOROZ', 'SIRA'], 'class_probabilities': {'aligned_to': 'class_order', 'finite': True, 'length': 7, 'row_sum': 1.0, 'type': 'array'}, 'decision_rule': 'argmax_class_score_or_probability', 'operational_prediction_available': False, 'predicted_class': {'allowed_values': ['SEKER', 'BARBUNYA', 'BOMBAY', 'CALI', 'DERMASON', 'HOROZ', 'SIRA'], 'type': 'string'}}


## 6. Fixed Demonstration Inputs

Demo values were sourced from a hash-validated, permitted training partition during implementation. The target label is intentionally omitted. This notebook does not read the training partition at runtime, does not retain a source row identifier, and does not compare predictions with truth.


In [7]:
DEMO_INPUTS = {
    "training-fixture-a": {
        "Area": 28734.0, "Perimeter": 638.018, "MajorAxisLength": 200.5247957,
        "MinorAxisLength": 182.7344194, "AspectRatio": 1.097356461,
        "Eccentricity": 0.411785251, "ConvexArea": 29172.0,
        "EquivDiameter": 191.2727505, "Extent": 0.783968133,
        "Solidity": 0.984985603, "Roundness": 0.887033637,
        "Compactness": 0.953860842, "ShapeFactor1": 0.006978659,
        "ShapeFactor2": 0.003563624, "ShapeFactor3": 0.909850506,
        "ShapeFactor4": 0.998430331,
    },
    "training-fixture-b": {
        "Area": 34327.0, "Perimeter": 677.311, "MajorAxisLength": 231.1581283,
        "MinorAxisLength": 189.3043718, "AspectRatio": 1.221092393,
        "Eccentricity": 0.573880789, "ConvexArea": 34714.0,
        "EquivDiameter": 209.0609812, "Extent": 0.793596116,
        "Solidity": 0.98885176, "Roundness": 0.940306539,
        "Compactness": 0.904406791, "ShapeFactor1": 0.006734003,
        "ShapeFactor2": 0.002779127, "ShapeFactor3": 0.817951644,
        "ShapeFactor4": 0.998794531,
    },
    "training-fixture-c": {
        "Area": 40111.0, "Perimeter": 740.571, "MajorAxisLength": 250.5138387,
        "MinorAxisLength": 204.0125498, "AspectRatio": 1.227933473,
        "Eccentricity": 0.580337086, "ConvexArea": 40644.0,
        "EquivDiameter": 225.9887417, "Extent": 0.760311623,
        "Solidity": 0.986886133, "Roundness": 0.919051716,
        "Compactness": 0.902100829, "ShapeFactor1": 0.006245515,
        "ShapeFactor2": 0.00255134, "ShapeFactor3": 0.813785907,
        "ShapeFactor4": 0.999274954,
    },
    "training-fixture-d": {
        "Area": 73927.0, "Perimeter": 1082.44, "MajorAxisLength": 397.7227442,
        "MinorAxisLength": 237.1305104, "AspectRatio": 1.677231427,
        "Eccentricity": 0.802820618, "ConvexArea": 74724.0,
        "EquivDiameter": 306.8008798, "Extent": 0.803886388,
        "Solidity": 0.989334083, "Roundness": 0.792876017,
        "Compactness": 0.771393852, "ShapeFactor1": 0.005379939,
        "ShapeFactor2": 0.001175065, "ShapeFactor3": 0.595048475,
        "ShapeFactor4": 0.998034003,
    },
}

single_input = deepcopy(DEMO_INPUTS["training-fixture-a"])
validated_single = normalize_inference_input(single_input, bundle=loaded_bundle)
display(validated_single.dataframe)


,Area,Perimeter,MajorAxisLength,MinorAxisLength,AspectRatio,Eccentricity,ConvexArea,EquivDiameter,Extent,Solidity,Roundness,Compactness,ShapeFactor1,ShapeFactor2,ShapeFactor3,ShapeFactor4
0,28734.0,638.018,200.524796,182.734419,1.097356,0.411785,29172.0,191.272751,0.783968,0.984986,0.887034,0.953861,0.006979,0.003564,0.909851,0.99843


## 7. Single-Row Inference and Seven-Class Probabilities


In [8]:
single_result = predict_multiclass(
    pipeline,
    single_input,
    bundle=loaded_bundle,
    runtime_report=runtime_report,
)
single_probability_table = pd.DataFrame({
    "class": single_result["class_order"],
    "probability": single_result["class_probabilities"],
})

print("Predicted class:", single_result["predicted_class"])
print("Top probability:", single_result["top_probability"])
print("Probability sum:", sum(single_result["class_probabilities"]))
display(single_probability_table)


Predicted class: SEKER
Top probability: 0.9977719092824935
Probability sum: 1.0


,class,probability
0,SEKER,0.997772
1,BARBUNYA,0.000015
2,BOMBAY,0.000005
3,CALI,0.000017
4,DERMASON,0.002051
5,HOROZ,0.000023
6,SIRA,0.000116


## 8. Batch Inference


In [9]:
batch_input = pd.DataFrame.from_dict(DEMO_INPUTS, orient="index")
batch_input.index.name = "case"
batch_core = predict_multiclass_batch(
    pipeline,
    batch_input,
    bundle=loaded_bundle,
    runtime_report=runtime_report,
)
batch_presentation = multiclass_output_to_frame(batch_core)
display(batch_presentation)


,predicted_class,probability_SEKER,probability_BARBUNYA,probability_BOMBAY,probability_CALI,probability_DERMASON,probability_HOROZ,probability_SIRA
case,,,,,,,,
training-fixture-a,SEKER,0.997772,0.000015,0.000005,0.000017,0.002051,0.000023,0.000116
training-fixture-b,SEKER,0.999304,0.000030,0.000008,0.000026,0.000353,0.000035,0.000244
training-fixture-c,SEKER,0.998978,0.000077,0.000011,0.000036,0.000502,0.000049,0.000347
training-fixture-d,BARBUNYA,0.000227,0.948848,0.000043,0.049938,0.000237,0.000477,0.000230


## 9. Representative Invalid Inputs

These examples deliberately show clear consumer errors. A broader negative matrix belongs in unit tests, not in this compact notebook.


In [10]:
invalid_inputs = {
    "missing-required-feature": {
        key: value for key, value in single_input.items() if key != loaded_bundle["feature_columns"][0]
    },
    "prohibited-target-present": {**single_input, loaded_bundle["target_column"]: "demo"},
    "nonnumeric-value": {**single_input, loaded_bundle["feature_columns"][0]: "not-a-number"},
    "missing-value": {**single_input, loaded_bundle["feature_columns"][8]: np.nan},
}

invalid_results = []
for case, invalid_input in invalid_inputs.items():
    try:
        predict_multiclass(pipeline, invalid_input, bundle=loaded_bundle)
    except InferenceInputError as exc:
        invalid_results.append({"case": case, "rejected": True, "message": str(exc)})
    else:
        raise AssertionError(f"Invalid demonstration input was accepted: {case}")

display(pd.DataFrame(invalid_results).set_index("case"))


,rejected,message
case,,
missing-required-feature,True,Missing required input columns: Area
prohibited-target-present,True,Prohibited input columns are present: Class
nonnumeric-value,True,Column Area contains 1 invalid numeric convers...
missing-value,True,Column Extent contains missing values.


## 10. Determinism, Column Ordering, and Single/Batch Consistency


In [11]:
repeated_single = predict_multiclass(
    pipeline,
    single_input,
    bundle=loaded_bundle,
    runtime_report=runtime_report,
)
reordered_batch = batch_input.loc[:, list(reversed(loaded_bundle["feature_columns"]))]
reordered_core = predict_multiclass_batch(
    pipeline,
    reordered_batch,
    bundle=loaded_bundle,
    runtime_report=runtime_report,
)

single_batch_probabilities = batch_core.iloc[0]["class_probabilities"]
assert repeated_single == single_result
assert single_result["predicted_class"] == batch_core.iloc[0]["predicted_class"]
assert single_result["class_order"] == batch_core.iloc[0]["class_order"]
assert np.allclose(single_result["class_probabilities"], single_batch_probabilities, rtol=0.0, atol=0.0)
assert batch_core["predicted_class"].tolist() == reordered_core["predicted_class"].tolist()
assert all(
    np.allclose(left, right, rtol=0.0, atol=0.0)
    for left, right in zip(
        batch_core["class_probabilities"],
        reordered_core["class_probabilities"],
        strict=True,
    )
)

print({
    "deterministic_repeated_inference": True,
    "single_batch_equivalence": True,
    "reordered_input_equivalence": True,
    "seven_class_alignment_validated": True,
})


{'deterministic_repeated_inference': True, 'single_batch_equivalence': True, 'reordered_input_equivalence': True, 'seven_class_alignment_validated': True}


## 11. Study Completion and Operational Limitations

Educational inference demonstration completed. The final-model handoff and inference bundle are independently consumable. Operational readiness remains unconfirmed: this notebook does not implement an API, deployment, monitoring, production data contract, SLOs, drift response, or retraining.


In [12]:
readiness_display = pd.Series({
    "final-model-handoff consumable": True,
    "inference-bundle consumable": True,
    "trusted model reload completed": True,
    "single-row inference completed": True,
    "batch inference completed": True,
    "seven-class alignment validated": True,
    "determinism validated": True,
    "operational_modeling_ready": loaded_handoff["operational_modeling_ready"],
    "operational_validity": loaded_handoff["operational_validity"],
    "api_implemented": loaded_handoff["api_implemented"],
}, name="status")
display(readiness_display.to_frame())


,status
final-model-handoff consumable,True
inference-bundle consumable,True
trusted model reload completed,True
single-row inference completed,True
batch inference completed,True
seven-class alignment validated,True
determinism validated,True
operational_modeling_ready,False
operational_validity,unconfirmed
api_implemented,False
